In [ ]:
from pathlib import Path
BASE_DIR = Path.cwd().parent
BULLETINS = BASE_DIR / "BULLETINS"
OUTPUT = BASE_DIR / "output"
DATA = BASE_DIR / "data"


In [ ]:

antidictionnaire = set()

seuilmin = 0.5

seuilmax = 20
with open(DATA/"tfxidf.txt", "r", encoding="utf8") as f:

    for ligne in f:

        doc, token, tfxidf = ligne.strip().split("\t")

        tfxidf = float(tfxidf)

        if tfxidf < seuilmin or tfxidf > seuilmax :
            antidictionnaire.add(token)


with open(DATA/"antidictionnaire.txt", "w", encoding="utf8") as f:

    for mot in antidictionnaire:
        f.write(mot + "\t\n") # on ajoute un vide après le mot parceque après dans la fonction substitue on utilisera ce vide pour remplacer le mot



In [ ]:
import re

def substitue(texte, fichier_substitution):

    subs = {}

    with open(fichier_substitution, "r", encoding="utf8") as f:

        for ligne in f:
            parties = ligne.strip().split("\t")
            mot = parties[0]

            if len(parties) > 1:
                remplace = parties[1]
            else:
                remplace = ""

            subs[mot] = remplace

    # découper le mot
    mots = re.findall(r"\b\w+\b", texte.lower())

    resultat = []

    for mot in mots:

        if mot in subs:

            if subs[mot] != "":
                resultat.append(subs[mot])
            # sinon supprimé

        else:
            resultat.append(mot)

    return " ".join(resultat)

In [ ]:
# Application de la fonction substitue au corpus 
from bs4 import BeautifulSoup
def corpus_filtrer1(corpus):
    # Lire le corpus XML
    with open(OUTPUT/corpus, "r", encoding="utf8") as f:
        contenu = f.read()

    #Parser le XML
    soup = BeautifulSoup(contenu, "html.parser")

    #Trouver tous les documents
    documents = soup.find_all("document")

    #Parcourir chaque document
    for doc in documents:

        # Liste des balises à nettoyer
        champs = ["texte", "titre" , "rubrique" , "legendeImage"]

        for champ in champs:

            balise = doc.find(champ)

            if balise:
                nouveau = substitue(balise.text, DATA/"antidictionnaire.txt")
                balise.string = nouveau

    # Sauvegarder le nouveau corpus
    with open(OUTPUT/"corpus_filtre.xml", "w", encoding="utf8") as f:
        f.write(str(soup))